# 01 Data Preparation
## 데이터 구성

### Analysis Unit
월별 공공 데이터 패널을 만들고, 정책 충격을 같은 월 단위로 맞춘다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "outputs" / "tables").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def load_table(name: str) -> pd.DataFrame:
    return pd.read_csv(TABLE_DIR / name)

def show_saved_figure(fig, name: str) -> None:
    path = FIGURE_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(path)))

In [2]:
panel = load_table("final_monthly_model_panel.csv")
policy = load_table("policy_calendar_monthly.csv")

panel["date"] = pd.to_datetime(panel["date"])
policy["date"] = pd.to_datetime(policy["date"])

overview = pd.DataFrame(
    [
        {
            "table": "model_panel",
            "rows": len(panel),
            "start": panel["date"].min().date(),
            "end": panel["date"].max().date(),
            "columns": panel.shape[1],
        },
        {
            "table": "policy_calendar",
            "rows": len(policy),
            "start": policy["date"].min().date(),
            "end": policy["date"].max().date(),
            "columns": policy.shape[1],
        },
    ]
)
overview

,table,rows,start,end,columns
0,model_panel,123,2016-01-01,2026-03-01,71
1,policy_calendar,87,2019-01-01,2026-03-01,9


패널은 2016년부터 2026년까지의 월별 서비스업 지표와 정책 강도를 같은 시간축에 올린다.

### Feasibility Decision
레스토랑 단독 장기 시계열이 없기 때문에 분석 대상은 숙박·음식 서비스 상대 반응으로 좁힌다.

In [3]:
feasibility = (
    panel[["estimand", "restaurant_hotel_gap_design_dropped"]]
    .drop_duplicates()
    .rename(
        columns={
            "estimand": "Estimand",
            "restaurant_hotel_gap_design_dropped": "Restaurant-hotel gap design dropped",
        }
    )
)
feasibility

,Estimand,Restaurant-hotel gap design dropped
0,accommodation-and-food-service relative response,True


따라서 이후 결과는 레스토랑 매출의 순수 효과가 아니라, AFS 부문의 상대 반응으로 해석한다.

### Policy Exposure
정책 기간은 월별 겹친 일수 비율로 계산한다.

In [4]:
policy_windows = (
    policy.loc[policy["policy_intensity_m"] > 0, ["period", "policy_intensity_m", "delivery_intensity_m", "active_events"]]
    .assign(active_events=lambda df: df["active_events"].fillna("policy"))
    .head(15)
)
policy_windows

,period,policy_intensity_m,delivery_intensity_m,active_events
21,2020-10,1.0000,0.0000,Khon La Khrueng Phase 1
22,2020-11,1.0000,0.0000,Khon La Khrueng Phase 1
23,2020-12,1.0000,0.0000,Khon La Khrueng Phase 1
24,2021-01,1.0000,0.0000,Khon La Khrueng Phase 2
25,2021-02,1.0000,0.0000,Khon La Khrueng Phase 2
26,2021-03,1.0000,0.0000,Khon La Khrueng Phase 2
30,2021-07,1.0000,0.0000,Khon La Khrueng Phase 3
31,2021-08,1.0000,0.0000,Khon La Khrueng Phase 3
32,2021-09,1.0000,0.0000,Khon La Khrueng Phase 3
33,2021-10,1.0000,0.0000,Khon La Khrueng Phase 3


월 단위 정책 강도는 정책이 걸친 기간을 과대·과소 계산하지 않기 위한 기준 변수다.

### Model Features
고노출 부문과 비교 부문, 정책 강도, 통제 변수를 최종 모델 테이블에 함께 둔다.

In [5]:
feature_groups = {
    "target_and_comparators": [
        "log_spi_afs",
        "log_low_equal_main",
        "log_low_synthetic_main",
        "gap_equal_main",
        "gap_synth_main",
    ],
    "policy": ["policy_intensity_m", "delivery_intensity_m", "policy_active", "delivery_active"],
    "controls": [
        "z_food_cpi_yoy",
        "z_core_cpi_yoy",
        "z_foreign_tourists_thousand",
        "z_accommodation_occupancy_rate_total",
        "z_private_consumption_index",
        "z_services_index",
    ],
}

pd.DataFrame(
    [{"group": group, "feature": feature} for group, features in feature_groups.items() for feature in features]
)

,group,feature
0,target_and_comparators,log_spi_afs
1,target_and_comparators,log_low_equal_main
2,target_and_comparators,log_low_synthetic_main
3,target_and_comparators,gap_equal_main
4,target_and_comparators,gap_synth_main
5,policy,policy_intensity_m
6,policy,delivery_intensity_m
7,policy,policy_active
8,policy,delivery_active
9,controls,z_food_cpi_yoy


핵심 테이블은 정책 충격, 비교 부문, 관광·물가·소비 통제를 함께 검증할 수 있게 구성되어 있다.